In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> def t1():\n'
                                       "...     p = ['Espresso', 'Birds of a Feather']\n"
                                       "...     out = queue_up(p, 'Not Like Us')\n"
                                       "...     return p == ['Espresso', 'Birds of a Feather', "
                                       "'Not Like Us'] and out is None\n"
                                       '>>> t1()\n'
                                       'True',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t1b():\n'
                                       '...     p = []\n'
                                       "...     queue_up(p, 'Good Luck, Babe!')\n"
                                       "...     return p == ['Good Luck, Babe!']\n"
                                       '>>> t1b()\n'
                                       'True',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 1,
        'suites': [{'cases': [{'code': '>>> q2_sort_returns is None\nTrue',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> unique_artists(['Sabrina', 'Chappell', 'Sabrina', "
                                       "'Tyla'])\n"
                                       "['Sabrina', 'Chappell', 'Tyla']",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> unique_artists(['Doja', 'Doja', 'Doja'])\n['Doja']",
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> unique_artists([])\n[]',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 3,
        'suites': [{'cases': [{'code': '>>> playlist_stats([3.5, 2.0, 4.25])\n'
                                       '(9.75, 2.0, 4.25)',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> playlist_stats([3.0])\n(3.0, 3.0, 3.0)',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> playlist_stats([]) == (0, 0, 0)\nTrue',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> def t5():\n'
                                       '...     data = [3.0, 1.0, 2.0]\n'
                                       '...     middle = median_length(data)\n'
                                       '...     return middle == 2.0 and data == [3.0, 1.0, '
                                       '2.0]\n'
                                       '>>> t5()\n'
                                       'True',
                               'hidden': False,
                               'locked': False,
                               'points': 1},
                              {'code': '>>> median_length([1.0, 2.0, 3.0, 4.0])\n2.5',
                               'hidden': False,
                               'locked': False,
                               'points': 1}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q6_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q7': {'name': 'q7',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q7_combined == ['Sunrise', 'Coffee', 'Neon', "
                                       "'Drive']\n"
                                       'True',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('11-Lists', _TESTS)
check = grader.check

print("Ready. 11-Lists loaded: 7 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: Lists — Playlist Manager

Every December your streaming app hands you a slideshow of numbers about your own
listening. Minutes played, top artists, your longest session. None of that is magic — it
is a **list** and a handful of loops.

In this homework you will build the pieces of a playlist manager: queueing tracks,
stripping duplicate artists, and working out the stats a wrapped screen shows.

Lists bring one genuinely new idea with them, and it is the one that catches people out:
lists are **mutable**. A string can never be changed — every string method hands you a new
string. A list *can* be changed in place, and several questions here are really about
noticing when that has happened.

## Question 1 — Queue a track

Write `queue_up(playlist, track)`, which adds `track` to the end of `playlist`.

Here is the part that matters: it **changes the list it was given**, and it **returns
nothing**.

```python
p = ['Espresso', 'Birds of a Feather']
queue_up(p, 'Not Like Us')
print(p)      # ['Espresso', 'Birds of a Feather', 'Not Like Us']
```

Because lists are mutable, a function that receives one receives the *actual* list, not a
copy — so changing it inside the function changes it for whoever called. That is why
`queue_up` does not need to return anything, and why it must not.

This mirrors how Python's own list methods behave: `.append()`, `.sort()` and `.reverse()`
all change the list in place and hand back `None`.

In [ ]:
def queue_up(playlist, track):
    ...

# try it out
tonight = ['Espresso', 'Birds of a Feather']
queue_up(tonight, 'Not Like Us')
print(tonight)

In [ ]:
grader.check("q1")

## Question 2 — The `None` that eats your playlist

This is the single most common list mistake in first year:

```python
plays = [4, 19, 2, 7]
plays = plays.sort()
print(plays)
```

The programmer wanted a sorted list. What they got was `None`, and their playlist is now
gone — because `.sort()` sorts the list **in place** and returns `None`, and that `None`
was then assigned over the top of the list.

Set `q2_sort_returns` to whatever `plays.sort()` evaluates to.

The fix, once you have seen it, is easy: call `plays.sort()` on its own line and keep
using `plays`. Or use the `sorted()` **function**, which leaves the original alone and
returns a new sorted list — you will need that one in Question 5.

In [ ]:
# a scratch cell to check yourself with
plays = [4, 19, 2, 7]
print(plays.sort())
print(plays)

q2_sort_returns = ...

In [ ]:
grader.check("q2")

## Question 3 — Who's actually on this playlist?

A playlist with nine tracks might only have four artists on it. Write
`unique_artists(artists)`, which takes a list of artist names — one per track, so names
repeat — and returns a new list with each artist **once**, in the order they first appear.

```
unique_artists(['Sabrina', 'Chappell', 'Sabrina', 'Tyla'])  ->  ['Sabrina', 'Chappell', 'Tyla']
unique_artists(['Doja', 'Doja', 'Doja'])                    ->  ['Doja']
unique_artists([])                                          ->  []
```

The order is part of the specification: `['Sabrina', 'Chappell', 'Tyla']` is right and
`['Chappell', 'Sabrina', 'Tyla']` is not.

Start with an empty list, loop over the artists, and use the `in` operator to ask whether
you have seen one already. `'Tyla' in result` is `True` or `False`, and reads exactly like
the English question you are asking.

In [ ]:
def unique_artists(artists):
    ...

# try it out
print(unique_artists(['Sabrina', 'Chappell', 'Sabrina', 'Tyla']))

In [ ]:
grader.check("q3")

## Question 4 — Wrapped stats

Write `playlist_stats(minutes)`. It takes a list of track lengths in minutes and returns a
**tuple** of three numbers, in this order:

```
(total, shortest, longest)
```

```
playlist_stats([3.5, 2.0, 4.25])  ->  (9.75, 2.0, 4.25)
playlist_stats([3.0])             ->  (3.0, 3.0, 3.0)
playlist_stats([])                ->  (0, 0, 0)
```

You do not need a loop for this. Python gives you `sum()`, `min()`, `max()` and `len()`,
and all four work on a list. Reach for them.

The empty playlist is the case worth thinking about: `min([])` raises a `ValueError`,
because the smallest of nothing is not a question with an answer. Handle it before you get
there.

Each of the three cases is worth 1 point.

In [ ]:
def playlist_stats(minutes):
    ...

# try it out
print(playlist_stats([3.5, 2.0, 4.25]))

In [ ]:
grader.check("q4")

## Question 5 — The median track length

The **median** is the middle value once the data is in order. With an even number of
values there is no single middle, so the median is the average of the two middle ones.

Write `median_length(minutes)`:

```
median_length([3.0, 1.0, 2.0])        ->  2.0    sorted: 1.0, 2.0, 3.0
median_length([1.0, 2.0, 3.0, 4.0])   ->  2.5    the average of 2.0 and 3.0
```

**Your function must not change the list it was given.** This is the trap: sorting is the
obvious first step, but `minutes.sort()` reorders the caller's playlist as a side effect,
and they did not ask you to do that. One of the two test cases checks specifically for
this.

You have two ways out — `sorted(minutes)`, which returns a new sorted list, or
`minutes.copy()` followed by sorting the copy. Either is fine. Knowing *why* you need one
of them is the point.

You will also want `//` to find the middle index, and `%` to ask whether the length is
odd or even.

In [ ]:
def median_length(minutes):
    ...

# try it out -- and check the original is untouched
lengths = [3.0, 1.0, 2.0]
print(median_length(lengths))
print(lengths)

In [ ]:
grader.check("q5")

## Question 6 — Two names, one playlist

```python
weekend = ['Espresso', 'Birds of a Feather']
party = weekend
party.append('Not Like Us')

print(len(weekend))
```

What does that print?

**A** — `2`, because `weekend` was never touched

**B** — `3`, because `party` and `weekend` are two names for the same list

**C** — it raises a `TypeError`

Set `q6_option` to `'A'`, `'B'` or `'C'`.

`party = weekend` does not copy anything. It gives the existing list a second name — an
**alias**. If you want a list you can change without disturbing the original, you have to
ask for one: `party = weekend.copy()`.

In [ ]:
q6_option = ...

In [ ]:
grader.check("q6")

## Question 7 — Stitch two playlists together

You have a morning playlist and an evening one, and you want a single list with the
morning tracks first.

Set `q7_combined` to that list — built from the two lists below with the `+` operator, not
typed out by hand.

Unlike `.append()`, the `+` operator does not change either list. It builds and returns a
**new** one, which is why you assign the result to something.

In [ ]:
mornings = ['Sunrise', 'Coffee']
evenings = ['Neon', 'Drive']

q7_combined = ...

print(q7_combined)
print(mornings, evenings)   # both should be unchanged

In [ ]:
grader.check("q7")

## Beyond the homework

Not graded:

- Write `top_artist(artists)`, returning the artist with the most tracks on the playlist.
  `.count()` will help.
- Write `shuffle(playlist)` using `random` — and decide, deliberately, whether yours
  shuffles in place like `.sort()` or returns a new list like `sorted()`. Document the
  choice in the docstring.
- Store each track as a small list, `[title, artist, minutes]`, and rewrite
  `playlist_stats` to work on that. What has to change?

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
